In [2]:
import gc
import polars as pl
from datetime import datetime, timezone

### <b>Global Variables.</b>

In [4]:
# Paths for BTCUSDT.
PATH_BINANCE_PERP_BTCUSDT_LOB       = r'data\binance_booktickers\perp_btcusdt.parquet'
PATH_BINANCE_PERP_BTCUSDT_LIQUID    = r'data\binance_liquidations\perp_btcusdt.parquet'
PATH_BYBIT_PERP_BTCUSDT_LIQUID      = r'data\bybit_liquidations\btcusdt.parquet'
PATH_BINANCE_PERP_BTCUSDT_TRADES    = r'data\binance_trades\perp_btcusdt.parquet'

# Paths for ETHUSDT.
PATH_BINANCE_PERP_ETHUSDT_LOB       = r'data\binance_booktickers\perp_ethusdt.parquet'
PATH_BINANCE_PERP_ETHUSDT_LIQUID    = r'data\binance_liquidations\perp_ethusdt.parquet'
PATH_BYBIT_PERP_ETHUSDT_LIQUID      = r'data\bybit_liquidations\ethusdt.parquet'
PATH_BINANCE_PERP_ETHUSDT_TRADES    = r'data\binance_trades\perp_ethusdt.parquet'

# All .parquet files.
PARQUET_FILES = {
    "binance_perp_btcusdt_lob": PATH_BINANCE_PERP_BTCUSDT_LOB,
    "binance_perp_btcusdt_liquid": PATH_BINANCE_PERP_BTCUSDT_LIQUID,
    "bybit_perp_btcusdt_liquid": PATH_BYBIT_PERP_BTCUSDT_LIQUID,
    "binance_perp_btcusdt_trades": PATH_BINANCE_PERP_BTCUSDT_TRADES,

    "binance_perp_ethusdt_lob": PATH_BINANCE_PERP_ETHUSDT_LOB,
    "binance_perp_ethusdt_liquid": PATH_BINANCE_PERP_ETHUSDT_LIQUID,
    "bybit_perp_ethusdt_liquid": PATH_BYBIT_PERP_ETHUSDT_LIQUID,
    "binance_perp_ethusdt_trades": PATH_BINANCE_PERP_ETHUSDT_TRADES,
}

## <b>Data Quality Check.</b>

We want to check quality of data:
1. Data completeness. 
2. Duplicates.
3. The monotony of timestamps.
4. Time coverage.
5. Gaps(time gaps).
6. Flow rate.
7. Validity of values. 
8. Summary table.

### <b>1.1. Data completneses.</b>

For each file, we count the number of null and nan values.

In [6]:
def check_data_completneses(
    lf: pl.LazyFrame,
    file_name: str
) -> pl.DataFrame:
    """
    Checks the completeness of data in the LazyFrame.
    
    For each column, it counts the number of NULL and NaN values (for float columns).
    Returns a table where each row represents a single column from the source file.
    """

    # Schema. Col name -> Data type.
    schema = lf.collect_schema()

    float_cols = [
        col for col, dtype in schema.items()
        if dtype in (pl.Float32, pl.Float64)
    ]

    # Number of null in each column.
    null_counts = (
        lf.null_count()
        .collect(streaming=True)
        .row(0, named=True)
    )

    # Number of NaN in each column.
    if float_cols:
        nan_counts = (
            lf
            .select(pl.col(float_cols).is_nan().sum())
            .collect(streaming=True)
            .row(0, named=True)
        )
    else:
        nan_counts = {}

    # Collect all results to one result.
    result = []

    for col, dtype in schema.items():
        result.append({
            "file": file_name,
            "column": col,
            "dtype": str(dtype),
            "null_count": null_counts[col],
            "nan_count": nan_counts.get(col, 0),
        })

    return pl.DataFrame(result)

In [7]:
completeness_tables = []

for name, path in PARQUET_FILES.items():

    # download parqut.
    lf = pl.scan_parquet(path)

    # calculate statistics.
    result = check_data_completneses(lf, name)

    # add result to results.
    completeness_tables.append(result)

    # clean ram.
    gc.collect()

# show result.
data_completeness = pl.concat(completeness_tables)
display(data_completeness)


C:\Users\Klimn\AppData\Local\Temp\ipykernel_21060\2510445819.py:22: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  lf.null_count()
C:\Users\Klimn\AppData\Local\Temp\ipykernel_21060\2510445819.py:30: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  lf


file,column,dtype,null_count,nan_count
str,str,str,i64,i64
"""binance_perp_btcusdt_lob""","""timestamp""","""Int64""",0,0
"""binance_perp_btcusdt_lob""","""ticker""","""String""",0,0
"""binance_perp_btcusdt_lob""","""bid_price""","""Float64""",0,0
"""binance_perp_btcusdt_lob""","""bid_amount""","""Float64""",0,0
"""binance_perp_btcusdt_lob""","""ask_price""","""Float64""",0,0
…,…,…,…,…
"""binance_perp_ethusdt_trades""","""timestamp""","""Int64""",0,0
"""binance_perp_ethusdt_trades""","""ticker""","""String""",0,0
"""binance_perp_ethusdt_trades""","""side""","""String""",0,0


### <b>1.2. Duplicates.</b> 

In [3]:
def check_duplicates(    
    lf: pl.LazyFrame,
    file_name: str
) -> pl.DataFrame:
    """
    Counts the number of exact duplicates in a Parquet file.
    """
    
    # Total number of elements.
    total = lf.select(pl.len()).collect().item()
    
    # Number of unique elements.
    n_unique = (
        lf
        .select(pl.struct(pl.all()).hash().alias("h"))
        .unique()
        .select(pl.len())
        .collect(streaming=True)
        .item()
    )
    
    return pl.DataFrame({
        "file": [file_name],
        "total_rows": [total],
        "full_duplicates": [total - n_unique],
    })

In [4]:
duplicates_tables = []

for name, path in PARQUET_FILES.items():

    # download parqut.
    lf = pl.scan_parquet(path)
    
    # calculate statistics.
    result = check_duplicates(lf, name)

    # add result to results.
    duplicates_tables.append(result)

    # clean ram.
    gc.collect()

# show result.
data_duplicates = pl.concat(duplicates_tables)
display(data_duplicates)


C:\Users\Klimn\AppData\Local\Temp\ipykernel_12924\2964240664.py:14: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  lf


file,total_rows,full_duplicates
str,i64,i64
"""binance_perp_btcusdt_lob""",99169477,0
"""binance_perp_btcusdt_liquid""",114255,0
"""bybit_perp_btcusdt_liquid""",228655,1
"""binance_perp_btcusdt_trades""",401902513,77816720
"""binance_perp_ethusdt_lob""",107797036,0
"""binance_perp_ethusdt_liquid""",131769,0
"""bybit_perp_ethusdt_liquid""",160214,2
"""binance_perp_ethusdt_trades""",705880385,138988014


#### <b>Duplicates Interim result.</b>

Trades show around 20% of duplicates for BTC and ETH.
I think there are two possible reasons:

1. Real artifacts - one transaction was recorded many times.
2. Order splitting - a large order is split into equal-sized parts in a single microsecond. These are valid trades.

Let's check that.

In [ ]:
# Chaeck for BTCUSDT.
lf = pl.scan_parquet(PATH_BINANCE_PERP_BTCUSDT_TRADES)

# Find the rows that are repeated.
(
    lf
    .group_by(pl.all())
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
    .head(10)
    .collect(streaming=True)
)

C:\Users\Klimn\AppData\Local\Temp\ipykernel_17984\819824549.py:5: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  lf


timestamp,ticker,side,price,amount,len
i64,str,str,f64,f64,u32
1764547452519000,"""perp:btcusdt""","""buy""",89594.3,0.023,175
1768921734070000,"""perp:btcusdt""","""sell""",90000.0,0.002,160
1769879289140000,"""perp:btcusdt""","""sell""",80000.0,0.002,158
1769441673196000,"""perp:btcusdt""","""buy""",88295.4,0.002,155
1769709511553000,"""perp:btcusdt""","""sell""",84000.0,0.002,154
1769437401038000,"""perp:btcusdt""","""buy""",87678.2,0.002,153
1772048359498000,"""perp:btcusdt""","""buy""",68850.0,0.003,152
1769014640797000,"""perp:btcusdt""","""buy""",87700.1,0.002,151
1770290699072000,"""perp:btcusdt""","""sell""",70000.0,0.002,149


In [ ]:
# Chaeck for ETHUSDT.
lf = pl.scan_parquet(PATH_BINANCE_PERP_ETHUSDT_TRADES)

# Find the rows that are repeated.
(
    lf
    .group_by(pl.all())
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
    .head(10)
    .collect(streaming=True)
)

#### <b>Duplicates Results.</b>

These are not duplicates, but legitimate transactions. 
Amount = 0.002 BTC repeated over 150 times in a single microsecond at a single price. 
This is a large order that the exchange has broken up.

### <b>1.3. The monotony of timestamps.</b>

Just make sure the timestamp isn't decreasing anywhere.

In [5]:
def check_monotonicity(    
    lf: pl.LazyFrame,
    file_name: str
) -> pl.DataFrame:
    """
    Checks for monotonicity of timestamps in a Parquet file.
    
    Identifies instances where the next timestamp is earlier than the previous one.
    Returns the number of violations and the maximum deviation (in microseconds).
    """
    
    # Create result, calculate, how many violation we have and find larget rollback.
    result = (
        lf
        .select(
            pl.col("timestamp").diff().alias("dt")
        )
        .filter(pl.col("dt") < 0)
        .select(
            pl.len().alias("violations"),
            pl.col("dt").min().alias("max_rollback_us"),
        )
        .collect(streaming=True)
    )
    
    violations = result["violations"].item()
    max_rollback = result["max_rollback_us"].item() if violations > 0 else 0
    
    return pl.DataFrame({
        "file": [file_name],
        "violations": [violations],
        "max_rollback_us": [max_rollback],
    })

In [7]:
mono_tables = []

for name, path in PARQUET_FILES.items():

    # download parqut.
    lf = pl.scan_parquet(path)
    
    # calculate statistics.
    result = check_monotonicity(lf, name)

    # append result to results.
    mono_tables.append(result)

    # clean ram.
    gc.collect()

display(pl.concat(mono_tables))

C:\Users\Klimn\AppData\Local\Temp\ipykernel_452\77266346.py:14: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  lf


file,violations,max_rollback_us
str,i64,i64
"""binance_perp_btcusdt_lob""",0,0
"""binance_perp_btcusdt_liquid""",0,0
"""bybit_perp_btcusdt_liquid""",63,-22000
"""binance_perp_btcusdt_trades""",0,0
"""binance_perp_ethusdt_lob""",0,0
"""binance_perp_ethusdt_liquid""",0,0
"""bybit_perp_ethusdt_liquid""",46,-16000
"""binance_perp_ethusdt_trades""",0,0


#### <b>Monotony Results.</b>

All Binance data is perfectly monotonic. Problems arise only during liquidations on Bybit:
- BTCUSDT: 63 violations, max rollback 22 ms.
- ETHUSDT: 46 violations, max rollback 16 ms.

I think, this is ok. We have only ~ tens of cases per hundred thousand lines, with a latency of around 22 ms. Given that “description.md” specifies that a +200 delay can be added to Bybit timestamps, 22 ms is irrelevant.

### <b>1.4. Time coverage.</b>

All files must start around 00:00 UTC on December 1, 2025, and end around 23:59 UTC on February 28, 2026.

In [8]:
def check_coverage(    
    lf: pl.LazyFrame,
    file_name: str
) -> pl.DataFrame:
    """
    Specifies the time range of the parquet file.
    
    Returns the minimum and maximum timestamps (in microseconds and in a human-readable format),
    as well as the number of rows.
    """

    result = (
        lf
        .select(
            pl.col("timestamp").min().alias("ts_min"),
            pl.col("timestamp").max().alias("ts_max"),
            pl.len().alias("rows"),
        )
        .collect(streaming=True)
    )
    
    ts_min = result["ts_min"].item()
    ts_max = result["ts_max"].item()
    
    # microseconds -> datetime.
    dt_min = datetime.fromtimestamp(ts_min / 1_000_000, tz=timezone.utc)
    dt_max = datetime.fromtimestamp(ts_max / 1_000_000, tz=timezone.utc)
    
    return pl.DataFrame({
        "file": [file_name],
        "rows": [result["rows"].item()],
        "ts_min": [ts_min],
        "ts_max": [ts_max],
        "dt_min": [str(dt_min)],
        "dt_max": [str(dt_max)],
    })

In [10]:
coverage_tables = []

for name, path in PARQUET_FILES.items():

    # download parqut.
    lf = pl.scan_parquet(path)

    # calculate statistics.
    result = check_coverage(lf, name)

    # append result to results.
    coverage_tables.append(result)

    # clean ram.
    gc.collect()

# show data.
display(pl.concat(coverage_tables))

C:\Users\Klimn\AppData\Local\Temp\ipykernel_452\3787224145.py:13: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  lf


file,rows,ts_min,ts_max,dt_min,dt_max
str,i64,i64,i64,str,str
"""binance_perp_btcusdt_lob""",99169477,1764547202075000,1772323199949000,"""2025-12-01 00:00:02.075000+00:…","""2026-02-28 23:59:59.949000+00:…"
"""binance_perp_btcusdt_liquid""",114255,1764547206091000,1772323150548000,"""2025-12-01 00:00:06.091000+00:…","""2026-02-28 23:59:10.548000+00:…"
"""bybit_perp_btcusdt_liquid""",228655,1764547202915000,1772323122271000,"""2025-12-01 00:00:02.915000+00:…","""2026-02-28 23:58:42.271000+00:…"
"""binance_perp_btcusdt_trades""",401902513,1764547200047000,1772323199992000,"""2025-12-01 00:00:00.047000+00:…","""2026-02-28 23:59:59.992000+00:…"
"""binance_perp_ethusdt_lob""",107797036,1764547200683000,1772323199829000,"""2025-12-01 00:00:00.683000+00:…","""2026-02-28 23:59:59.829000+00:…"
"""binance_perp_ethusdt_liquid""",131769,1764547209455000,1772322590541000,"""2025-12-01 00:00:09.455000+00:…","""2026-02-28 23:49:50.541000+00:…"
"""bybit_perp_ethusdt_liquid""",160214,1764547207779000,1772321662075000,"""2025-12-01 00:00:07.779000+00:…","""2026-02-28 23:34:22.075000+00:…"
"""binance_perp_ethusdt_trades""",705880385,1764547200016000,1772323199993000,"""2025-12-01 00:00:00.016000+00:…","""2026-02-28 23:59:59.993000+00:…"


#### <b>Time Coverage Results.</b>

All 8 files start on 2025-12-01 00:00 and end on February 2026-02-28:
- The liquidations end a few minutes early -> no one was liquidated.
- Trades and LOB are covered up to 23:59:59.

### <b>1.5. Gaps(time gaps).</b>

In [11]:
def check_gaps(    
    lf: pl.LazyFrame,
    file_name: str
) -> pl.DataFrame:
    
    """
    Analyzes the intervals between events in a LazyFrame.
    
    Returns the median, p99, and maximum gap (in seconds),
    as well as the number of pauses longer than 1 and 5 minutes
    """
    

    result = (
        lf
        .select(
            (pl.col("timestamp").diff() / 1_000_000)
            .alias("dt_sec")
        )
        .filter(pl.col("dt_sec").is_not_null())
        .select(
            pl.col("dt_sec").median().alias("median_gap"),
            pl.col("dt_sec").quantile(0.99).alias("p99_gap"),
            pl.col("dt_sec").max().alias("max_gap"),
            (pl.col("dt_sec") > 60).sum().alias("gaps_over_1min"),
            (pl.col("dt_sec") > 300).sum().alias("gaps_over_5min"),
        )
        .collect(streaming=True)
    )
    
    return pl.DataFrame({
        "file": [file_name],
        "median_gap_sec": [round(result["median_gap"].item(), 6)],
        "p99_gap_sec": [round(result["p99_gap"].item(), 3)],
        "max_gap_sec": [round(result["max_gap"].item(), 1)],
        "gaps_over_1min": [result["gaps_over_1min"].item()],
        "gaps_over_5min": [result["gaps_over_5min"].item()],
    })

In [13]:
gap_tables = []

for name, path in PARQUET_FILES.items():

    # download parqut.
    lf = pl.scan_parquet(path)
    
    # calculate statistics.
    result = check_gaps(lf, name)

    # append result to results.
    gap_tables.append(result)

    # clean ram.
    gc.collect()

# show results.
display(pl.concat(gap_tables))

C:\Users\Klimn\AppData\Local\Temp\ipykernel_452\539139170.py:15: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  lf


file,median_gap_sec,p99_gap_sec,max_gap_sec,gaps_over_1min,gaps_over_5min
str,f64,f64,f64,i64,i64
"""binance_perp_btcusdt_lob""",0.052,0.368,199.3,4,0
"""binance_perp_btcusdt_liquid""",4.042,1035.246,5676.1,21539,6923
"""bybit_perp_btcusdt_liquid""",0.059,902.536,32396.3,11212,5326
"""binance_perp_btcusdt_trades""",0.0,0.45,46.4,0,0
"""binance_perp_ethusdt_lob""",0.052,0.316,354.0,4,1
"""binance_perp_ethusdt_liquid""",4.247,825.377,6812.2,24474,6826
"""bybit_perp_ethusdt_liquid""",0.081,1243.486,19957.5,11221,5437
"""binance_perp_ethusdt_trades""",0.0,0.281,46.3,0,0


#### <b>Gaps(time gaps) Results.</b>
- Trades: Median gap ~ 0, max 46 seconds, no pauses longer than a minute.
- LOB: Median 52 ms, only 4 pauses lasting > 1 min, max 3–6 min. Most likely brief pauses by the exchange. 
- Liquidations: gaps are to be expected here. Liquidations only occur during periods of high volatility.

Interesting moment:
- Binance liquidations: median gap ~ 4 seconds, relatively rare.
- Bybit liquidations: median gap ~ 60–80 ms - occur many times more frequently.

Main Point: The difference in the frequency of liquidations between Bybit and Binance should be taken into account when building features. It’s probably best to compare metrics similar to Lopez de Prado’s Dollar Bars (price * amount).

### <b>1.6. Flow rate.</b>

This check will provide an overview: average, minimum, and maximum rates by day. 

In [14]:
def check_frequency(    
    lf: pl.LazyFrame,
    file_name: str
) -> pl.DataFrame:
    """
    Calculates the average event rate in a LazyFrame.
    
    Groups by day and returns the average, minimum, and maximum number
    of events per day, as well as the average number per second.
    """

    result = (
        lf
        .select(
            pl.col("timestamp"),
            # Hour UTC.
            ((pl.col("timestamp") / 1_000_000) % 86400 / 3600)
            .floor()
            .cast(pl.Int32)
            .alias("hour"),
            # Day.
            (pl.col("timestamp") / 1_000_000 / 86400)
            .floor()
            .cast(pl.Int64)
            .alias("day"),
        )
        .group_by("day")
        .agg(pl.len().alias("events"))
        .select(
            pl.col("events").mean().alias("avg_per_day"),
            pl.col("events").min().alias("min_per_day"),
            pl.col("events").max().alias("max_per_day"),
        )
        .collect(streaming=True)
    )
    
    avg = result["avg_per_day"].item()
    
    return pl.DataFrame({
        "file": [file_name],
        "avg_per_day": [round(avg)],
        "avg_per_sec": [round(avg / 86400, 1)],
        "min_per_day": [result["min_per_day"].item()],
        "max_per_day": [result["max_per_day"].item()],
    })

In [16]:
freq_tables = []

for name, path in PARQUET_FILES.items():

    # download parqut.
    lf = pl.scan_parquet(path)
    
    # calculate statistics.
    result = check_frequency(lf, name)

    # append result to results.
    freq_tables.append(result)

    # clean ram.
    gc.collect()

# show results.
display(pl.concat(freq_tables))

C:\Users\Klimn\AppData\Local\Temp\ipykernel_452\229941781.py:13: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  lf


file,avg_per_day,avg_per_sec,min_per_day,max_per_day
str,i64,f64,i64,i64
"""binance_perp_btcusdt_lob""",1101883,12.8,682900,1533586
"""binance_perp_btcusdt_liquid""",1270,0.0,166,5650
"""bybit_perp_btcusdt_liquid""",2541,0.0,44,24310
"""binance_perp_btcusdt_trades""",4465583,51.7,599776,19580992
"""binance_perp_ethusdt_lob""",1197745,13.9,798605,1572412
"""binance_perp_ethusdt_liquid""",1464,0.0,238,5313
"""bybit_perp_ethusdt_liquid""",1780,0.0,59,14032
"""binance_perp_ethusdt_trades""",7843115,90.8,1071409,30864956


#### <b>Flow rate Results.</b>

- Trades: ETH is more active than BTC. The maximum daily volume is 4–5 times higher than the average - these are days with high volatility.

- LOB:  BTC and ETH are roughly the same (~13 updates/sec).

- Liquidations: The finding from gaps is confirmed: Bybit reports ~2x more events than Binance (2,541 vs. 1,270 for BTC). The min/max range is huge: on quiet days, there are almost no liquidations (44–238), while on volatile days, there are thousands and tens of thousands. This indicates strong clustering.


### <b>1.7.Validity of values.</b>

Looking for anomalies in the data.

In [17]:
def check_values(    
    lf: pl.LazyFrame,
    file_name: str,
    file_type: str
) -> pl.DataFrame:
    """
    Checks the validity of values in the parquet file.
    
    - For LOB: bid/ask price > 0, amount > 0, bid < ask.
    - For trades/liquidations: price > 0, amount > 0, side set to [buy, sell].
    """
    
    
    if file_type == "lob":
        result = (
            lf
            .select(
                (pl.col("bid_price") <= 0).sum().alias("bad_bid_price"),
                (pl.col("ask_price") <= 0).sum().alias("bad_ask_price"),
                (pl.col("bid_amount") <= 0).sum().alias("bad_bid_amount"),
                (pl.col("ask_amount") <= 0).sum().alias("bad_ask_amount"),
                (pl.col("bid_price") >= pl.col("ask_price")).sum().alias("crossed_spread"),
                pl.len().alias("rows"),
            )
            .collect(streaming=True)
        )
        return pl.DataFrame({
            "file": [file_name],
            "rows": [result["rows"].item()],
            "bad_price": [result["bad_bid_price"].item() + result["bad_ask_price"].item()],
            "bad_amount": [result["bad_bid_amount"].item() + result["bad_ask_amount"].item()],
            "bad_side": [0],
            "crossed_spread": [result["crossed_spread"].item()],
        })
    
    else:  # trades of liquidations
        result = (
            lf
            .select(
                (pl.col("price") <= 0).sum().alias("bad_price"),
                (pl.col("amount") <= 0).sum().alias("bad_amount"),
                (~pl.col("side").is_in(["buy", "sell"])).sum().alias("bad_side"),
                pl.len().alias("rows"),
            )
            .collect(streaming=True)
        )
        return pl.DataFrame({
            "file": [file_name],
            "rows": [result["rows"].item()],
            "bad_price": [result["bad_price"].item()],
            "bad_amount": [result["bad_amount"].item()],
            "bad_side": [result["bad_side"].item()],
            "crossed_spread": [0],
        })

In [18]:
def get_file_type(name: str) -> str:
    if "lob" in name:
        return "lob"
    return "trades_or_liq"

val_tables = []

for name, path in PARQUET_FILES.items():

    
    # download parqut.
    lf = pl.scan_parquet(path)
    
    # calculate statistics.
    ftype = "lob" if "lob" in name else "other"
    result = check_values(lf, name, ftype)

    # add result to results.
    val_tables.append(result)

    # clean ram.
    gc.collect()

# show results.
display(pl.concat(val_tables))

C:\Users\Klimn\AppData\Local\Temp\ipykernel_452\3778505828.py:16: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  lf
C:\Users\Klimn\AppData\Local\Temp\ipykernel_452\3778505828.py:38: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  lf


file,rows,bad_price,bad_amount,bad_side,crossed_spread
str,i64,i64,i64,i64,i64
"""binance_perp_btcusdt_lob""",99169477,0,0,0,0
"""binance_perp_btcusdt_liquid""",114255,0,0,0,0
"""bybit_perp_btcusdt_liquid""",228655,0,0,0,0
"""binance_perp_btcusdt_trades""",401902513,0,0,0,0
"""binance_perp_ethusdt_lob""",107797036,0,0,0,0
"""binance_perp_ethusdt_liquid""",131769,0,0,0,0
"""bybit_perp_ethusdt_liquid""",160214,0,0,0,0
"""binance_perp_ethusdt_trades""",705880385,0,0,0,0


#### <b>Validity of values Results.</b>
Perfectly clean data - zero anomalies across all files. No negative prices, zero amounts, invalid sides, or overlapping spreads.

### **Data Quality Summary.**

| Check | Result | Notes |
|---|---|---|
| **Completeness** | Pass | Zero NULL and NaN values across all 8 files. |
| **Duplicates** | Pass | LOB and liquidations: 0–2 full duplicates. Trades: ~20% repeated rows are order splitting, not artifacts. |
| **Timestamp monotonicity** | Pass | Binance: perfectly monotonic. Bybit liquidations: ~50 violations (max rollback 22 ms) - negligible given +200 ms cross-exchange delay. |
| **Time coverage** | Pass | All files cover 2025-12-01 -> 2026-02-28. Liquidation files end a few minutes early, that's ok.|
| **Gaps** | Pass | Trades/LOB: continuous, no gaps > 1 min. Liquidation gaps are expected (no liquidations during calm markets). |
| **Flow rate** | Pass | Trades: BTC ~52/sec, ETH ~91/sec. LOB: ~13/sec. Bybit reports ~2x more liquidation events than Binance - compare by volume (price × amount), not count. |
| **Value validity** | Pass | Zero invalid prices, amounts, sides, or crossed spreads across all files. |

**Key takeaways for signal construction:**
- Data is clean, no preprocessing or filtering needed.
- Bybit liquidations should be sorted by timestamp and shifted by +200 ms before use.
- Bybit reports finer-grained liquidations than Binance - use notional volume (price × amount) for cross-exchange features, not event count. Similar to Lopez de Prado's Dollar Bars approach, where activity is measured in dollar terms rather than tick count to normalize across different market microstructures.
- Liquidations are highly clustered (min/max daily count ratio up to 500x) - this clustering is the core property to exploit in the filter signal.